# 🤖 OpenAI API로 LLM 다뤄보기

**생성형 AI 개발 · 4차시 실습 — API 기반 생성형 AI 서비스 기초**

---

이 노트북에서는 여러분이 직접 **OpenAI API 키**를 입력하고,
**프롬프트를 작성**해서 LLM의 답변을 받아봅니다.

### 오늘 배우는 것
| 단계 | 내용 |
|---|---|
| 1 | 라이브러리 설치 & API 키 입력 |
| 2 | 첫 요청 보내기 (Hello, GPT!) |
| 3 | 내 프롬프트로 질문하기 |
| 4 | `system` 메시지로 역할 부여 (role prompting) |
| 5 | zero-shot · few-shot 프롬프팅 |
| 6 | `temperature` · `max_tokens` 파라미터 실험 |
| 7 | 멀티턴 대화 만들기 |
| 8 | 실습 과제 |

> 💡 **API 키가 필요합니다.** https://platform.openai.com/api-keys 에서 발급받으세요.
> 키는 `sk-...` 형태이며, **절대 남에게 공유하거나 코드에 그대로 적어두지 마세요.**


## 1단계 · 준비하기

먼저 OpenAI 공식 파이썬 라이브러리를 설치합니다. (이미 설치돼 있으면 그냥 넘어갑니다.)

In [1]:
# OpenAI 라이브러리 설치
%pip install openai --quiet

# 설치 후 정상적으로 불러와지는지 확인
try:
    import openai
    print("✅ 설치 완료 · openai", openai.__version__)
except ImportError:
    print("⚠️ 불러오기 실패 → 상단 메뉴 [런타임] ▸ [런타임 다시 시작] 후 이 셀을 다시 실행하세요.")

✅ 설치 완료 · openai 2.43.0


### API 키 입력

아래 셀을 실행하면 입력창이 뜹니다. 거기에 여러분의 API 키를 붙여넣으세요.
`getpass`를 쓰기 때문에 키가 화면에 **보이지 않게** 안전하게 입력됩니다.

In [2]:
import os
import getpass

# 입력한 키는 화면에 표시되지 않습니다.
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요 (sk-...): ")

print("✅ API 키가 등록되었습니다.")

OpenAI API Key를 입력하세요 (sk-...): ··········
✅ API 키가 등록되었습니다.


### 클라이언트 만들기

`client`는 우리가 OpenAI에게 요청을 보내는 **통로**입니다.
위에서 등록한 환경변수(`OPENAI_API_KEY`)를 자동으로 읽어옵니다.

> ⚠️ **모델 선택 주의**
> - `gpt-4o-mini` · `gpt-4.1-mini` : 일반(비추론형) 모델 → 이 실습 코드가 **그대로 동작**합니다. (추천)
> - `gpt-5.4-mini` · `gpt-5.4-nano` 등 **GPT-5 계열은 추론형**이라 `temperature`를 바꿀 수 없고(1 고정),
>   `max_tokens` 대신 `max_completion_tokens`를 써야 합니다. (아래 `ask()` 함수가 알아서 처리하지만,
>   6단계의 temperature 비교 실습 결과는 달라지지 않습니다.)

In [3]:
from openai import OpenAI

client = OpenAI()   # 환경변수에서 API 키를 자동으로 읽어옵니다

# 사용할 모델 이름 (원하면 아래 중 하나로 바꿔보세요)
#   "gpt-4o-mini"    → 가장 저렴한 일반 모델 (실습에 추천 · 기본값)
#   "gpt-4.1-mini"   → 조금 더 성능이 좋은 일반 모델
#   "gpt-5.4-mini"   → 최신 추론형 모델 (temperature 고정 주의)
#   "gpt-5.4"        → 상위 추론형 모델 (비용 높음)
MODEL = "gpt-4o-mini"

print("✅ 클라이언트 준비 완료 · 사용 모델:", MODEL)

✅ 클라이언트 준비 완료 · 사용 모델: gpt-4o-mini


## 2단계 · 첫 요청 보내기

LLM에게 메시지를 보내는 기본 형태는 다음과 같습니다.

```python
client.chat.completions.create(
    model=MODEL,          # 어떤 모델을 쓸지
    messages=[            # 대화 내용 (list)
        {"role": "user", "content": "질문 내용"}
    ]
)
```

- `messages`는 **대화 목록**입니다. `role`이 `"user"`면 사람, `"assistant"`면 GPT, `"system"`이면 역할 지시입니다.
- 답변 텍스트는 `response.choices[0].message.content` 안에 들어 있습니다.
- 답변 길이(`max_tokens`)·창의성(`temperature`) 같은 옵션은 6단계에서 다룹니다.

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "안녕하세요! 당신을 한 문장으로 소개해 주세요."}
    ]
)

# 답변 텍스트 꺼내기
print(response.choices[0].message.content)

안녕하세요! 저는 다양한 주제에 대해 정보 제공과 질문에 답변할 수 있는 인공지능 언어 모델입니다.


### 응답 안에는 뭐가 들어있을까?

답변 말고도 **몇 개의 토큰을 썼는지** 같은 정보가 함께 옵니다.
토큰은 대략적인 '단어 조각' 단위이고, 요금이 토큰 수에 따라 매겨지므로 알아두면 좋습니다.

In [5]:
print("입력 토큰 수 :", response.usage.prompt_tokens)
print("출력 토큰 수 :", response.usage.completion_tokens)
print("멈춘 이유    :", response.choices[0].finish_reason)

입력 토큰 수 : 20
출력 토큰 수 : 27
멈춘 이유    : stop


## 3단계 · 내 프롬프트로 질문하기 ✍️

매번 긴 코드를 쓰면 번거로우니, **질문을 넣으면 답변을 돌려주는 함수**로 감싸겠습니다.
이제부터는 `ask("질문")` 한 줄이면 됩니다.

> `system` 인자를 주면 대화 맨 앞에 역할 지시 메시지가 추가됩니다.
> GPT-5 계열 모델도 에러 없이 돌아가도록 파라미터 이름을 자동으로 맞춰 줍니다.

In [ ]:
# 메세지 형식
'''
client.chat.completions.create(
    model="gpt-4o-mini",          # ← 최상위 인자
    temperature=1.0,              # ← 최상위 인자
    max_tokens=1024,              # ← 최상위 인자
    messages=[                    # ← 최상위 인자 (리스트)
        {"role": "system", "content": "너는 친절한 튜터야."},   # 원소 1 (딕셔너리)
        {"role": "user",   "content": "재귀가 뭐야?"},          # 원소 2 (딕셔너리)
    ]
)
'''

In [6]:
def ask(prompt, system=None, max_tokens=1024, temperature=1.0):
    """프롬프트를 보내고 LLM의 답변 텍스트를 돌려줍니다."""
    messages = []
    if system:                                       # 역할(system) 지시가 있으면 맨 앞에 추가
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = {"model": MODEL, "messages": messages}
    if MODEL.startswith("gpt-5"):                     # GPT-5 계열(추론형)은 파라미터가 다름
        kwargs["max_completion_tokens"] = max_tokens  #  · max_tokens 대신 이것을 사용
        #  · temperature는 기본값(1)만 지원하므로 넣지 않음
    else:                                            # gpt-4o / gpt-4.1 계열(일반)
        kwargs["max_tokens"] = max_tokens
        kwargs["temperature"] = temperature

    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

print("✅ ask() 함수 준비 완료")

✅ ask() 함수 준비 완료


👇 **여기가 핵심입니다.** 아래 따옴표 안의 프롬프트를 **여러분이 원하는 질문으로 바꿔가며** 실행해 보세요.

In [7]:
my_prompt = "파이썬 리스트와 튜플의 차이를 초보자에게 설명해줘. 예시 코드도 하나 보여줘."

print(ask(my_prompt))

파이썬에서 리스트(list)와 튜플(tuple)는 둘 다 여러 개의 값을 저장할 수 있는 자료구조입니다. 하지만 몇 가지 중요한 차이점이 있습니다. 

### 리스트(list)
- **변경 가능(mutable)**: 리스트의 요소를 추가하거나 제거할 수 있습니다. 즉, 리스트는 한 번 생성한 후에도 내용을 수정할 수 있습니다.
- **대괄호([])**: 리스트는 대괄호로 생성합니다.

### 튜플(tuple)
- **변경 불가능(immutable)**: 튜플은 한 번 생성하면 내용이 변경될 수 없습니다. 요소를 추가하거나 삭제할 수 없습니다.
- **소괄호(())**: 튜플은 소괄호로 생성합니다.

### 예시 코드

```python
# 리스트 예제
my_list = [1, 2, 3]
print("원래 리스트:", my_list)

# 리스트 요소 추가
my_list.append(4)
print("요소 추가 후 리스트:", my_list)

# 리스트 요소 삭제
my_list.remove(2)
print("요소 삭제 후 리스트:", my_list)

# 튜플 예제
my_tuple = (1, 2, 3)
print("원래 튜플:", my_tuple)

# 튜플은 변경할 수 없음
# my_tuple.append(4)  # 이 줄은 오류를 발생시킵니다.
# my_tuple.remove(2)  # 이 줄 또한 오류를 발생시킵니다.

# 튜플의 내용을 출력
print("변경할 수 없는 튜플:", my_tuple)
```

### 요약
- 리스트와 튜플은 둘 다 여러 값을 저장할 수 있지만, 리스트는 변경할 수 있고 튜플은 변경할 수 없습니다.
- 리스트는 대괄호로 생성하고, 튜플은 소괄호로 생성합니다. 

이렇게 리스트와 튜플의 가장 큰 차이점과 사용 예시를 통해 이해할 수 있었으면 좋겠습니다!


> 🔁 위 셀의 `my_prompt` 내용을 자유롭게 바꿔서 여러 번 실행해 보세요.
> 예) `"오늘 점심 메뉴 3개만 추천해줘"`, `"재귀함수를 비유로 설명해줘"`, `"이 문장을 영어로 번역해줘: 생성형 AI는 재미있다"`

## 4단계 · 역할 부여하기 (System Message · Role Prompting)

`system` 메시지는 LLM에게 **"너는 이런 역할이야"** 라고 성격·말투·규칙을 정해주는 지시입니다.
같은 질문이라도 역할에 따라 답변이 완전히 달라집니다.

In [ ]:
question = "재귀(recursion)가 뭐야?"

print("=== 역할: 엄격한 교수님 ===")
print(ask(question, system="너는 컴퓨터공학과 교수다. 정확한 용어를 사용해 격식 있게 설명한다."))

print("\n=== 역할: 친근한 선배 ===")
print(ask(question, system="너는 다정한 학과 선배다. 반말로 친근하게, 쉬운 비유를 들어 설명한다."))

=== 역할: 엄격한 교수님 ===
재귀(Recursion)는 함수가 자기 자신을 호출하는 프로그래밍 기법을 의미합니다. 이는 문제를 해결하기 위해 문제를 더 작은 형태로 분할하고, 동일한 방식으로 해결하는 접근 방법입니다. 대개 재귀 함수는 두 가지 기본 요소를 포함합니다: 기준 사례(base case)와 재귀 경우(recursive case)입니다.

1. **기준 사례**: 재귀 호출이 종료되는 조건으로, 더 이상 자기 자신을 호출하지 않고 결과를 반환하는 부분입니다. 이는 재귀 호출이 무한 루프에 빠지지 않도록 하는 중요한 요소입니다.

2. **재귀 경우**: 문제를 보다 간단한 형태로 변환하여 자신을 다시 호출하는 부분입니다. 재귀 호출을 통해 원래의 문제를 조금씩 해결해 나가는 방식입니다.

재귀의 대표적인 예로는 팩토리얼 계산, 피보나치 수열 생성, 트리 탐색 등이 있습니다. 예를 들어, 팩토리얼을 재귀적으로 정의하면 아래와 같습니다:

\[
n! = 
\begin{cases} 
1 & \text{if } n = 0 \\ 
n \times (n - 1)! & \text{if } n > 0 
\end{cases}
\]

재귀 함수는 개념적으로 간단하고 문제를 명확하게 표현할 수 있는 장점이 있지만, 잘못 구현할 경우 스택 오버플로우(stack overflow)와 같은 문제가 발생할 수 있으므로 주의가 필요합니다. 이와 함께 재귀는 반복문(iteration)을 사용한 방법에 비해 성능 면에서 비효율적일 수 있으며, 메모리 사용량이 더 클 수 있다는 점도 고려해야 합니다.

=== 역할: 친근한 선배 ===
재귀(recursion)는 어떤 함수가 자기 자신을 호출하는 프로그래밍 기법이야. 쉽게 비유하자면, 거울에 비친 너를 보는 것과 비슷해. 거울 앞에 서면 너의 모습이 거울에 비치고, 그 거울 속에도 또 다른 너의 모습이 비치고, 이렇게 계속 반복되는 느낌이야.

예를 들어, 계단을 올라갈 때 생각해보자. 너가 1층에서 2층으로 가고 싶어서 "나는 

> 🔁 `system` 내용을 바꿔서 **해적 말투**, **초등학생도 알아듣게**, **한 줄 요약만** 등 다양하게 시켜 보세요.

## 5단계 · Zero-shot vs Few-shot 프롬프팅

- **Zero-shot**: 예시 없이 그냥 시키는 것
- **Few-shot**: **예시 몇 개를 보여준 뒤** 같은 방식으로 하라고 시키는 것 → 형식·스타일을 더 잘 따라옵니다.

In [8]:
# Zero-shot : 예시 없이 바로 요청
zero_shot = "다음 문장의 감정을 '긍정' 또는 '부정'으로 답해줘: 이 영화 정말 시간 아까웠어."
print("[Zero-shot]")
print(ask(zero_shot))

[Zero-shot]
부정


In [9]:
# Few-shot : 예시를 먼저 보여주고 같은 형식으로 답하게 하기
few_shot = """다음 예시처럼 문장의 감정을 분류해줘.

문장: 오늘 날씨가 정말 좋아서 기분이 최고야!
감정: 긍정

문장: 버스를 놓쳐서 지각했어. 최악이야.
감정: 부정

문장: 이 식당 음식은 정말 훌륭했어.
감정:"""

print("[Few-shot]")
print(ask(few_shot))

[Few-shot]
긍정


> 💡 Few-shot은 **원하는 출력 형식을 예시로 고정**하고 싶을 때 특히 강력합니다.
> (예: 항상 JSON으로 답하게 하기, 항상 3줄 요약만 하게 하기)

## 6단계 · 파라미터 실험하기

두 가지 자주 쓰는 값을 조절해 봅시다.

| 파라미터 | 의미 | 값이 클 때 | 값이 작을 때 |
|---|---|---|---|
| `temperature` | 답변의 **무작위성/창의성** (0 ~ 2) | 다양·창의적 | 일관·정확 |
| `max_tokens` | 답변의 **최대 길이** | 길게 가능 | 짧게 잘림 |

> ⚠️ 이 실습은 **일반 모델(gpt-4o-mini 등)** 기준입니다.
> GPT-5 계열은 `temperature`가 1로 고정되어 두 결과가 거의 같게 나옵니다.

In [10]:
prompt = "AI를 주제로 짧은 삼행시를 지어줘."

print("=== temperature = 0.0 (거의 항상 비슷) ===")
print(ask(prompt, temperature=0.0))

print("\n=== temperature = 1.0 (다양하게) ===")
print(ask(prompt, temperature=1.0))

=== temperature = 0.0 (거의 항상 비슷) ===
인공지능, 미래를 열어  
지혜의 바다를 헤엄쳐  
능력의 한계를 넘어가네.

=== temperature = 1.0 (다양하게) ===
인간의 꿈을 담은  
지혜의 파도, 넘실대며  
새로운 세상 열어가네.


In [11]:
# max_tokens 를 작게 주면 답변이 도중에 잘립니다.
print("=== max_tokens = 30 (짧게 잘림) ===")
print(ask("인공지능의 역사를 설명해줘.", max_tokens=30))

=== max_tokens = 30 (짧게 잘림) ===
인공지능(AI)의 역사는 20세기 중반으로 거슬러 올라가며, 여러 핵심 사건과 발전을 거


## 7단계 · 멀티턴 대화 (대화 기억하기)

LLM은 기본적으로 **이전 대화를 기억하지 못합니다.**
기억하게 하려면 지금까지의 대화 전체를 `messages` 리스트에 담아 매번 함께 보내야 합니다.
`user` → `assistant` → `user` → ... 순서로 번갈아 쌓습니다.

In [ ]:
# 대화 기록을 담을 리스트
history = []

def chat(user_message):
    """이전 대화를 기억하며 이어서 대화합니다."""
    history.append({"role": "user", "content": user_message})

    kwargs = {"model": MODEL, "messages": history}
    if MODEL.startswith("gpt-5"):                 # GPT-5 계열은 파라미터 이름이 다름
        kwargs["max_completion_tokens"] = 1024
    else:
        kwargs["max_tokens"] = 1024

    response = client.chat.completions.create(**kwargs)
    answer = response.choices[0].message.content
    history.append({"role": "assistant", "content": answer})  # 답변도 기록에 추가
    return answer

print("✅ chat() 함수 준비 완료")

✅ chat() 함수 준비 완료


In [ ]:
print(chat("내 이름은 코알라야. 기억해줘."))

안녕하세요, 코알라님! 반갑습니다. 어떻게 도와드릴까요?


In [ ]:
# 앞에서 이름을 알려줬으니, 이번엔 기억하고 있어야 합니다.
print(chat("내 이름이 뭐라고 했지?"))

당신의 이름은 코알라입니다!


> 🔁 `chat("...")` 을 계속 실행하며 대화를 이어가 보세요.
> 대화를 처음부터 다시 시작하려면 아래 셀로 기록을 비웁니다.

In [15]:
history.clear()
print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")

NameError: name 'history' is not defined

## 8단계 · 실습 과제 🎯

아래 빈칸을 채워 직접 만들어 보세요. 정답은 하나가 아닙니다!

**과제 1.** `system` 메시지를 사용해 **"항상 이모지를 섞어 답하는 여행 가이드"** 를 만들고,
"부산 여행 코스를 추천해줘" 라고 물어보세요.

**과제 2.** Few-shot 예시를 만들어, 한글 단어를 넣으면 **영어 단어로만** 답하는 번역기를 만들어 보세요.

**과제 3.** `chat()` 함수로 3턴 이상 대화를 이어가며, LLM이 앞 내용을 기억하는지 확인해 보세요.

In [13]:
# 과제 1 예시 (여기서부터 직접 수정해 보세요)
guide_system = "너는 쾌할하고 이국적인 여행 가이드다. 답변에 관련 이모지를 자연스럽게 섞어 사용한다."

print(ask("일본 오키나와 여행 코스를 추천해줘.", system=guide_system))

오키나와는 맑은 바다와 아름다운 자연으로 유명하죠! 🌊✨ 다음은 오키나와에서 즐길 수 있는 멋진 여행 코스입니다.

### 1일차: 나하 탐방
- **시티센터 탐방** 🏙️: 나하의 국제 거리(国際通り)에서 쇼핑과 스트리트 푸드를 즐겨보세요. 🍜✨
- **슈리성** 방문 🏯: 유네스코 세계유산으로 지정된 이곳에서 오키나와의 역사와 문화를 배워보세요. 🏰
- **저녁식사** 🍣: 현지 맛집에서 오키나와 소바(沖縄そば)를 즐기세요!

### 2일차: 해양 액티비티
- **카귀 시마**(嘉手納)로 이동 🚤: 수영, 스노클링, 제트스키 등 다양한 해양 액티비티를 즐길 수 있는 곳입니다. 🐠🌴
- **점심**: 바닷가 근처에서 신선한 해산물을 맛보세요. 🦐🍚
- **석양 감상** 🌅: 해변에서 아름다운 석양을 감상하세요.

### 3일차: 자연 탐방
- **만자모**(万座毛) 방문 🌄: 아름다운 절벽과 푸른 바다가 절경을 이루는 곳입니다. 사진명소로 유명해요! 📸💙
- **오키나와 월드** 🌿: 전통문화 체험 및 동굴 탐험을 할 수 있는 장소입니다.
- **저녁식사**: 오키나와 전통 요리인 고야 참프루(ゴーヤーチャンプルー)를 먹어보세요! 🍲😋

### 4일차: 숨겨진 보물 찾기
- **구니가미 섬**(国頭) 탐방: 조용하고 한적한 해변에서 여유로운 시간을 보내세요. 🏖️
- **오키나와 민속촌**: 전통적인 오키나와 생활 방식을 엿볼 수 있습니다. 🎭
- **마지막 저녁**: 낚지(たこ) 요리로 마무리하고, 여행의 추억을 나눠보세요! 🌌🥂

이 코스를 통해 오키나와의 자연과 문화를 최대한으로 즐기세요! 즐거운 여행 되세요~! 🌏✈️


##예시 1번 결과
부산은 매력적인 해변과 맛있는 음식, 그리고 멋진 풍경이 가득한 도시랍니다! 🌊✨ 여기 부산에서의 하루 여행 코스를 추천해드릴게요.

### 아침
1. **해운대 해수욕장** 🏖️
   - 아침 일찍 해운대 해수욕장에 놀러 가세요! 시원한 바다 바람을 느끼며 산책하기 좋습니다.
   - 근처에서 맛있는 **해물파전**이나 **회덮밥**으로 아침 식사도 즐길 수 있어요. 🍽️

### 오전
2. **동백섬** 🌳
   - 해운대 해수욕장에서 가까워요. 동백섬을 걸으며 아름다운 자연을 만끽하세요.
   - **더홀리데이** 카페에서 커피 한 잔도 잊지 마세요! ☕️

### 점심
3. **부산 전통시장** 🛍️
   - **자갈치 시장** 또는 **자갈치 시장**에서 신선한 해산물을 즐기세요!
   - 구운 생선이나 해물 찜이 인기랍니다! 🍤

### 오후
4. **감천문화마을** 🎨
   - 감천문화마을로 가서 색색의 집과 예술 작품을 감상하세요. 사진 찍기 좋은 장소가 정말 많아요! 📸
   - 마을을 한 바퀴 돌아보면서 예쁜 카페도 들러보세요. ☕️

### 저녁
5. **광안리 해수욕장** 🌅
   - 저녁에는 광안리 해수욕장에서 멋진 일몰을 감상해보세요.
   - 근처의 바다 전망 레스토랑에서 **회**나 **해물탕**을 즐기면서 저녁 식사를 즐기세요. 🍲

### 밤
6. **광안대교 야경** 🌉
   - 저녁 식사 후에는 광안대교의 야경을 감상하며 산책을 즐겨 보세요. 조명이 정말 아름답답니다! 🌟

부산에서 멋진 하루 보내세요! 🥳

##예시 1번 수정 결과

오키나와는 맑은 바다와 아름다운 자연으로 유명하죠! 🌊✨ 다음은 오키나와에서 즐길 수 있는 멋진 여행 코스입니다.

### 1일차: 나하 탐방
- **시티센터 탐방** 🏙️: 나하의 국제 거리(国際通り)에서 쇼핑과 스트리트 푸드를 즐겨보세요. 🍜✨
- **슈리성** 방문 🏯: 유네스코 세계유산으로 지정된 이곳에서 오키나와의 역사와 문화를 배워보세요. 🏰
- **저녁식사** 🍣: 현지 맛집에서 오키나와 소바(沖縄そば)를 즐기세요!

### 2일차: 해양 액티비티
- **카귀 시마**(嘉手納)로 이동 🚤: 수영, 스노클링, 제트스키 등 다양한 해양 액티비티를 즐길 수 있는 곳입니다. 🐠🌴
- **점심**: 바닷가 근처에서 신선한 해산물을 맛보세요. 🦐🍚
- **석양 감상** 🌅: 해변에서 아름다운 석양을 감상하세요.

### 3일차: 자연 탐방
- **만자모**(万座毛) 방문 🌄: 아름다운 절벽과 푸른 바다가 절경을 이루는 곳입니다. 사진명소로 유명해요! 📸💙
- **오키나와 월드** 🌿: 전통문화 체험 및 동굴 탐험을 할 수 있는 장소입니다.
- **저녁식사**: 오키나와 전통 요리인 고야 참프루(ゴーヤーチャンプルー)를 먹어보세요! 🍲😋

### 4일차: 숨겨진 보물 찾기
- **구니가미 섬**(国頭) 탐방: 조용하고 한적한 해변에서 여유로운 시간을 보내세요. 🏖️
- **오키나와 민속촌**: 전통적인 오키나와 생활 방식을 엿볼 수 있습니다. 🎭
- **마지막 저녁**: 낚지(たこ) 요리로 마무리하고, 여행의 추억을 나눠보세요! 🌌🥂

이 코스를 통해 오키나와의 자연과 문화를 최대한으로 즐기세요! 즐거운 여행 되세요~! 🌏✈️

## 과제 2 · 과제 3 은 아래 빈 셀에서 자유롭게 작성해 보세요.

In [14]:
#과제2
few_shot = """다음 예시처럼 한글 단어를 넣으면 영어 단어로만 번역하여 답해줘

단어: 제품
영단어: If translated -> product

단어: 바다
영단어: If translated -> ocean

단어: 건물.
영단어: """

print("[Few_shot]")
print(ask(few_shot))

[Few_shot]
If translated -> building


In [16]:
#과제3

# 대화 기록을 담을 리스트
new_history = []

def new_chat(user_message):
    """이전 대화를 기억하며 이어서 대화합니다."""
    new_history.append({"role": "user", "content": user_message})

    kwargs = {"model": MODEL, "messages": new_history}
    if MODEL.startswith("gpt-5"):                 # GPT-5 계열은 파라미터 이름이 다름
        kwargs["max_completion_tokens"] = 1024
    else:
        kwargs["max_tokens"] = 1024

    response = client.chat.completions.create(**kwargs)
    answer = response.choices[0].message.content
    new_history.append({"role": "assistant", "content": answer})  # 답변도 기록에 추가
    return answer

print("✅ new_chat() 함수 준비 완료")

✅ new_chat() 함수 준비 완료


###테스트

In [ ]:
#대화기록 초기화
new_history.clear()
print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")

In [17]:
print(new_chat("안녕, 내 이름은 앤리시고 게임 기획자야."))

안녕하세요, 앤리시! 게임 기획자로서 어떤 프로젝트를 작업하고 계신가요? 관심 있는 게임 장르나 아이디어가 있다면 함께 이야기해볼까요?


In [18]:
print(new_chat("나는 오픈월드,액션 장르의 스팀펑크 세계관의 게임을 개발중이야."))

스팀펑크 세계관의 오픈월드 액션 게임이라니, 정말 흥미로운 프로젝트네요! 스팀펑크는 독특한 비주얼과 기계적 요소를 가져와서 다양한 창의적인 아이디어를 실현할 수 있는 기회를 제공합니다. 

현재 구상하고 있는 게임의 주요 요소나 특징에 대해 좀 더 이야기해주실 수 있을까요? 스토리, 캐릭터, 게임 메커니즘 등 어떤 부분이 특히 중요한지 궁금합니다!


In [19]:
print(new_chat("주요 특징 중에서 특별히 얘기할 수 있는 아이디어는 전투 로직이야. 단순한 RPG의 스킬 발동형이 아닌 QTE의 외부 오브젝트 상호작용 시스템을 활용해 공격하는 방식이야."))

재미있는 전투 로직 아이디어네요! QTE(Quick Time Event)를 활용한 외부 오브젝트 상호작용 시스템은 플레이어의 몰입감을 높일 수 있을 것 같습니다. 이를 통해 전투가 더 역동적이고 다채롭게 변할 수 있겠네요.

이 시스템을 통해 전투에서 플레이어가 어떤 방식으로 오브젝트와 상호작용하게 될지 구체적으로 구상해 보셨나요? 예를 들어, 특정 오브젝트를 활용하여 적을 공격하거나 방어하는 방식일까요? 또한, 플레이어가 환경을 활용해 전투를 유리하게 가져가도록 어떻게 유도할 계획인지 궁금합니다.


In [20]:
print(new_chat("내가 누구고 무엇을 하고 있는지 잘 알게 된 것 같아?"))

네, 앤리시! 당신은 게임 기획자로서 스팀펑크 세계관의 오픈월드 액션 게임을 개발 중인 것 같습니다. 특별히 전투 로직으로 QTE 기반의 외부 오브젝트 상호작용 시스템을 도입하려고 하신다고 하셨죠. 매우 창의적이고 흥미로운 접근입니다!

더 구체적인 아이디어나 고민하고 있는 부분이 있다면 언제든지 나눠주세요. 도움이 필요하시다면 더 많은 논의도 가능하니, 편하게 말씀해 주세요!


In [ ]:
#대화기록 삭제
new_history.clear()
print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")

---

## 🎉 수고하셨습니다!

오늘 배운 것을 정리하면:

1. `client.chat.completions.create(...)` 로 LLM에 요청을 보낸다
2. `messages` 는 `system` / `user` / `assistant` 로 이루어진 **대화 목록**이다
3. `system` 으로 **역할**을, `temperature`·`max_tokens` 로 **답변 스타일과 길이**를 조절한다
4. **few-shot 예시**로 원하는 출력 형식을 유도할 수 있다
5. 대화를 기억시키려면 **이전 기록을 매번 함께 보낸다**

다음 시간에는 이 API를 **FastAPI 백엔드**와 연동해 실제 챗봇 서비스로 확장합니다. 🚀

> ⚠️ **보안 주의:** API 키가 노출되면 즉시 https://platform.openai.com/api-keys 에서 삭제·재발급하세요.
